# Atividade 4: contagem de moedas com YOLO11 e YOLO + ViT

Antes de rodar: ativar a GPU (T4) em Ambiente de execução > Alterar tipo de ambiente.

Sem moedas para fotografar, o dataset é sintético: `gerar_dataset.py` cola recortes de fotos reais de moedas
(`moedas_base/`, Wikimedia Commons, domínio público ou CC0) sobre fundos de mesa, com rotação, sombra, luz e
tamanho variados. São 200 imagens com 5 a 40 moedas cada: 140 de treino, 40 de validação e 20 de teste
(as fotos de teste final). Como o gerador sabe onde pôs cada moeda, as caixas e o total em reais são exatos.

As classes são por valor e face (`5c_valor`, `5c_efigie`, ..., `1real_efigie`). A versão com uma classe
por valor (`5c`, ..., `1real`) é gerada aqui a partir dela, sem anotar de novo.

In [ ]:
!pip install -q ultralytics datasets

VALORES = {"5c": 0.05, "10c": 0.10, "25c": 0.25, "50c": 0.50, "1real": 1.00}
CLASSES = list(VALORES)  # classes por valor, na ordem do dicionário

## Parte A: dados

In [ ]:
# Gera o dataset sintético (classes por valor e face) com o gerador do repositório
!git clone -q --depth 1 https://github.com/data-science-unicamp/aplicacoes-data-science /content/repo
!cd /content/repo/atividade4 && python gerar_dataset.py
!rm -rf /content/por_face && cp -r /content/repo/atividade4/dataset /content/por_face

In [ ]:
# Cria a versão "por valor": copia o dataset e troca cada classe "10c_valor" ou "10c_efigie" por "10c"
import os, glob, shutil, yaml

face = yaml.safe_load(open("/content/por_face/data.yaml"))
shutil.copytree("/content/por_face", "/content/por_valor", dirs_exist_ok=True)
for lbl in glob.glob("/content/por_valor/*/labels/*.txt"):
    linhas = []
    for linha in open(lbl).read().splitlines():
        c, *v = linha.split()
        linhas.append(" ".join([str(CLASSES.index(face["names"][int(c)].split("_")[0]))] + v))
    open(lbl, "w").write("\n".join(linhas))

# data.yaml dos dois datasets com caminhos absolutos
for nome, names in [("por_face", face["names"]), ("por_valor", CLASSES)]:
    root = f"/content/{nome}"
    cfg = {"train": f"{root}/train/images", "val": f"{root}/valid/images", "test": f"{root}/test/images",
           "nc": len(names), "names": names}
    yaml.safe_dump(cfg, open(f"{root}/data.yaml", "w"))
    print(nome, cfg["names"])

In [ ]:
# Quantidade de imagens e de moedas por classe em cada conjunto (para o relatório)
import pandas as pd
from collections import Counter

tabela = {}
for split in ["train", "valid", "test"]:
    cont = Counter()
    for lbl in glob.glob(f"/content/por_face/{split}/labels/*.txt"):
        cont.update(face["names"][int(l.split()[0])] for l in open(lbl).read().splitlines())
    cont["imagens"] = len(glob.glob(f"/content/por_face/{split}/images/*"))
    tabela[split] = cont
pd.DataFrame(tabela).fillna(0).astype(int)

## Parte A: treino (fine-tuning)

Dois modelos `yolo11s` partindo dos pesos do COCO, um para cada definição de classes. `imgsz=960` porque as
moedas são objetos pequenos. Augmentation no treino: além do padrão do Ultralytics (brilho, contraste, escala,
espelhamento horizontal e mosaico), rotação de até 180 graus e espelhamento vertical, já que moeda não tem
orientação fixa.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

modelos = {}
for nome in ["por_face", "por_valor"]:
    model = YOLO("yolo11s.pt")
    model.train(data=f"/content/{nome}/data.yaml", epochs=100, imgsz=960, batch=16, patience=20,
                degrees=180, flipud=0.5, project="/content/runs", name=nome, exist_ok=True)
    modelos[nome] = YOLO(f"/content/runs/{nome}/weights/best.pt")
    display(Image(f"/content/runs/{nome}/results.png", width=900))  # curvas de perda e métricas

## Parte A: avaliação no conjunto de teste

In [ ]:
# mAP, precisão e revocação no teste e matriz de confusão de cada modelo
metricas = {}
for nome, model in modelos.items():
    m = model.val(data=f"/content/{nome}/data.yaml", split="test", imgsz=960,
                  project="/content/runs", name=f"{nome}_teste", exist_ok=True)
    metricas[nome] = m.box.map50
    print(f"{nome}: mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f} | "
          f"precisão {m.box.mp:.3f} | revocação {m.box.mr:.3f}")
    display(Image(f"/content/runs/{nome}_teste/confusion_matrix.png", width=700))

In [ ]:
# Avaliação pelo valor em reais: compara cada foto com as caixas anotadas (o total real)
import time, torch
from PIL import Image as PILImage
from torchvision.ops import box_iou

def rotulos(p):
    # caixas anotadas da foto (x1 y1 x2 y2 em pixels) e o valor de cada moeda
    w, h = PILImage.open(p).size
    caixas, classes = [], []
    for linha in open(p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt").read().splitlines():
        c, xc, yc, bw, bh = linha.split()[:5]
        xc, yc, bw, bh = float(xc) * w, float(yc) * h, float(bw) * w, float(bh) * h
        caixas.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
        classes.append(face["names"][int(c)].split("_")[0])
    return caixas, classes

def avaliar(prever, split):
    # prever(foto) -> (caixas, classes por valor); devolve acurácia por moeda, erro médio no total e tempo
    acertos = n = 0
    fotos = []
    for p in sorted(glob.glob(f"/content/por_face/{split}/images/*")):
        t = time.perf_counter()
        caixas, classes = prever(p)
        ms = (time.perf_counter() - t) * 1000
        reais, classes_reais = rotulos(p)
        if caixas:  # cada moeda anotada casa com a caixa prevista de maior IoU (mínimo 0.5)
            iou = box_iou(torch.tensor(reais), torch.tensor(caixas))
            for i, c in enumerate(classes_reais):
                j = int(iou[i].argmax())
                if iou[i, j] >= 0.5:
                    acertos += classes[j] == c
                    iou[:, j] = 0
        n += len(classes_reais)
        previsto = sum(VALORES[c] for c in classes)
        real = sum(VALORES[c] for c in classes_reais)
        fotos.append({"foto": os.path.basename(p), "real": real, "previsto": previsto,
                      "erro": abs(previsto - real), "ms": ms})
    fotos = pd.DataFrame(fotos)
    return {"acuracia": acertos / n, "erro_medio": fotos["erro"].mean(),
            "ms": fotos["ms"].iloc[1:].mean(), "fotos": fotos}  # tempo ignora a 1a foto (aquecimento)

def prever_yolo(model, conf):
    def prever(p):
        r = model.predict(p, conf=conf, imgsz=960, verbose=False)[0]
        return r.boxes.xyxy.tolist(), [model.names[int(c)].split("_")[0] for c in r.boxes.cls]
    return prever

def escolher_conf(fabrica):
    # o limiar de confiança muda o total: escolhe o de menor erro no conjunto de VALIDAÇÃO
    erros = {conf: avaliar(fabrica(conf), "valid")["erro_medio"] for conf in [0.25, 0.4, 0.5, 0.6, 0.7]}
    print("erro médio (R$) na validação por conf:", {k: round(v, 3) for k, v in erros.items()})
    return min(erros, key=erros.get)

resultados, confs = {}, {}
for nome, model in modelos.items():
    confs[nome] = escolher_conf(lambda conf: prever_yolo(model, conf))
    resultados[nome] = avaliar(prever_yolo(model, confs[nome]), "test")
    print(f"{nome}: conf {confs[nome]} | acurácia por moeda {resultados[nome]['acuracia']:.3f} | "
          f"erro médio no total R$ {resultados[nome]['erro_medio']:.2f}")
    display(resultados[nome]["fotos"])

## Parte B: YOLO + ViT

O YOLO "por valor" é usado só para localizar as moedas; cada caixa é recortada com 10% de margem e um
Vision Transformer com fine-tuning classifica o recorte em 5c, 10c, 25c, 50c ou 1real.

In [ ]:
# Recortes das caixas anotadas em pastas por classe: recortes/train/5c/..., recortes/val/5c/...
def com_margem(b, tamanho, m=0.10):
    x1, y1, x2, y2 = b
    dx, dy = (x2 - x1) * m, (y2 - y1) * m
    return (max(0, x1 - dx), max(0, y1 - dy), min(tamanho[0], x2 + dx), min(tamanho[1], y2 + dy))

for split, destino in [("train", "train"), ("valid", "val")]:
    for p in glob.glob(f"/content/por_face/{split}/images/*"):
        img = PILImage.open(p).convert("RGB")
        for k, (b, c) in enumerate(zip(*rotulos(p))):
            os.makedirs(f"/content/recortes/{destino}/{c}", exist_ok=True)
            img.crop(com_margem(b, img.size)).save(f"/content/recortes/{destino}/{c}/{os.path.basename(p)}_{k}.jpg")

for destino in ["train", "val"]:
    print(destino, {c: len(os.listdir(f"/content/recortes/{destino}/{c}")) for c in CLASSES})

In [ ]:
# Fine-tuning do ViT; no treino, augmentation com rotação e variação de brilho e contraste
import numpy as np
from datasets import load_dataset
from torchvision import transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer

ds = load_dataset("imagefolder", data_dir="/content/recortes")
labels = ds["train"].features["label"].names
ckpt = "google/vit-base-patch16-224-in21k"
proc = AutoImageProcessor.from_pretrained(ckpt)
aug = transforms.Compose([transforms.RandomRotation(180), transforms.ColorJitter(0.3, 0.3)])

def prep(batch, treino=False):
    ims = [im.convert("RGB") for im in batch["image"]]
    out = proc([aug(im) for im in ims] if treino else ims, return_tensors="pt")
    out["labels"] = batch["label"]
    return out

ds["train"].set_transform(lambda b: prep(b, treino=True))
ds["validation"].set_transform(prep)

clf = AutoModelForImageClassification.from_pretrained(
    ckpt, num_labels=len(labels), id2label=dict(enumerate(labels)), label2id={l: i for i, l in enumerate(labels)})

args = TrainingArguments("/content/vit-moedas", per_device_train_batch_size=16, num_train_epochs=8,
                         learning_rate=3e-5, eval_strategy="epoch", save_strategy="no",
                         remove_unused_columns=False, report_to="none")
trainer = Trainer(model=clf, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
                  data_collator=lambda b: {"pixel_values": torch.stack([x["pixel_values"] for x in b]),
                                           "labels": torch.tensor([x["labels"] for x in b])},
                  compute_metrics=lambda p: {"acuracia": (p.predictions.argmax(-1) == p.label_ids).mean()})
trainer.train()
trainer.save_model("/content/vit-moedas")
proc.save_pretrained("/content/vit-moedas")

In [ ]:
# Pipeline completo: YOLO localiza, recorta com margem, ViT classifica e soma
det = modelos["por_valor"]
clf = clf.eval()

def prever_hibrido(conf):
    def prever(p):
        img = PILImage.open(p).convert("RGB")
        caixas = det.predict(img, conf=conf, imgsz=960, verbose=False)[0].boxes.xyxy.tolist()
        if not caixas:
            return [], []
        inp = proc([img.crop(com_margem(b, img.size)) for b in caixas], return_tensors="pt").to(clf.device)
        with torch.no_grad():
            preds = clf(**inp).logits.argmax(-1).tolist()
        return caixas, [clf.config.id2label[i] for i in preds]
    return prever

confs["hibrido"] = escolher_conf(prever_hibrido)
resultados["hibrido"] = avaliar(prever_hibrido(confs["hibrido"]), "test")
print(f"hibrido: conf {confs['hibrido']} | acurácia por moeda {resultados['hibrido']['acuracia']:.3f} | "
      f"erro médio no total R$ {resultados['hibrido']['erro_medio']:.2f}")
display(resultados["hibrido"]["fotos"])

# mAP50 da Parte B: só a detecção conta, então mede o YOLO como detector de classe única
metricas["hibrido"] = det.val(data="/content/por_valor/data.yaml", split="test", imgsz=960, single_cls=True,
                              project="/content/runs", name="detector_teste", exist_ok=True).box.map50

## Comparação nas fotos de teste final

In [ ]:
nomes = {"por_valor": "A: YOLO (1 classe por valor)", "por_face": "A: YOLO (classe por face)",
         "hibrido": "B: YOLO + ViT"}
pd.DataFrame([{"Solução": nomes[k], "mAP@0.5": round(metricas[k], 3),
               "Acurácia por moeda": round(resultados[k]["acuracia"], 3),
               "Erro médio no total (R$)": round(resultados[k]["erro_medio"], 2),
               "Tempo (ms/img)": round(resultados[k]["ms"], 1), "conf": confs[k]} for k in nomes])

## Análise de erros e demonstração

Cada foto de teste final com as caixas previstas, o total previsto e o total real, da que mais errou para a
que menos errou (modelo por face).

In [ ]:
import cv2

nome = "por_face"
model = modelos[nome]
for _, f in resultados[nome]["fotos"].sort_values("erro", ascending=False).iterrows():
    r = model.predict(f"/content/por_face/test/images/{f['foto']}", conf=confs[nome], imgsz=960, verbose=False)[0]
    img = r.plot(labels=True, conf=False)
    cv2.putText(img, f"previsto R$ {f['previsto']:.2f} | real R$ {f['real']:.2f}", (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 0, 255), 4)
    cv2.imwrite("/content/tmp.jpg", img)
    print(f["foto"])
    display(Image("/content/tmp.jpg", width=600))

In [ ]:
# Baixa os pesos (best.pt de cada YOLO e a pasta do ViT), os gráficos e o dataset gerado num zip
from google.colab import files

!cd /content && zip -qr runs.zip runs vit-moedas por_face
files.download("/content/runs.zip")